# Creating and loading calendars

There are two main ways to work with calendars:

1. construct a `Calendar(...)` directly from holidays and nonworking weekdays
2. load a calendar with `Calendar.load(...)`

Use the constructor when your application already owns the holiday data. Use `load` when you want packaged calendars or a JSON file.

## Calendars that come with the package

The package ships with these calendar names:

- `ANBIMA`
- `B3`
- `Actual`

In [ ]:
from bizdays import Calendar, list_calendars

{name: Calendar.load(name=name).name for name in ["ANBIMA", "B3", "Actual"]}

In [ ]:
list_calendars()

`list_calendars()` is the programmatic discovery helper for calendar sources.

It returns a dictionary with:

- `packaged`: the curated calendars shipped with `bizdays`
- `pandas_market_calendars`: metadata about the optional PMC integration
- `exchange_calendars`: metadata about the optional exchange calendar integration
- `workalendar`: metadata about the optional workalendar integration

Each integration entry contains:

- `available`: whether the optional dependency is installed
- `prefix`: the prefix used with `Calendar.load(name="<prefix><calendar>")`
- `calendars`: the discovered calendar names when available, otherwise an empty list

In [ ]:
info = list_calendars()
{
    "packaged": info["packaged"],
    "pmc_available": info["pandas_market_calendars"]["available"],
    "pmc_prefix": info["pandas_market_calendars"]["prefix"],
    "xcal_available": info["exchange_calendars"]["available"],
    "xcal_prefix": info["exchange_calendars"]["prefix"],
    "work_available": info["workalendar"]["available"],
    "work_prefix": info["workalendar"]["prefix"],
}

`Actual` is the simplest packaged calendar: it has no holidays and no nonworking weekdays, so every day is a business day.

In [ ]:
actual = Calendar.load(name="Actual")
actual.isbizday(["2024-01-01", "2024-01-06", "2024-01-07"])

## Creating a calendar from scratch with `Calendar(...)`

The constructor accepts:

- `holidays`: ISO date strings or date-like objects
- `weekdays`: names of nonworking weekdays
- `startdate` and `enddate`: optional calendar bounds
- `name`: a display name
- `financial`: whether to use financial counting rules

In [ ]:
custom = Calendar(
    holidays=["2024-01-01", "2024-04-21", "2024-12-25"],
    weekdays=["Saturday", "Sunday"],
    startdate="2024-01-01",
    enddate="2024-12-31",
    name="Example",
    financial=True,
)
custom

In [ ]:
{
    "name": custom.name,
    "weekdays": custom.weekdays,
    "startdate": str(custom.startdate),
    "enddate": str(custom.enddate),
    "holidays": [str(day) for day in custom.holidays],
}

If you omit `startdate` and `enddate`, they are inferred from the holiday range. If you construct a calendar without holidays, a broad default range is used.

In [ ]:
weekends_only = Calendar(
    weekdays=["Saturday", "Sunday"],
    name="WeekendsOnly",
)
{
    "startdate": str(weekends_only.startdate),
    "enddate": str(weekends_only.enddate),
    "isbizday": weekends_only.isbizday(["2024-01-05", "2024-01-06", "2024-01-07"]),
}

## JSON calendar layout

`Calendar.load(filename=...)` expects a JSON object with this layout:

- `name`: string
- `weekdays`: list of strings
- `holidays`: list of ISO date strings
- `financial`: boolean
- `adjust.from`: optional string accepted for schema compatibility
- `adjust.to`: optional string accepted for schema compatibility

The current Python implementation builds the calendar from `name`, `weekdays`, `holidays`, and `financial`.

In [ ]:
payload = {
    "name": "CustomJson",
    "weekdays": ["saturday", "sunday"],
    "holidays": ["2024-01-01", "2024-04-21", "2024-12-25"],
    "financial": True,
    "adjust.from": "following",
    "adjust.to": "preceding",
}
payload

In [ ]:
import json
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmpdir:
    filename = Path(tmpdir) / "custom-calendar.json"
    filename.write_text(json.dumps(payload), encoding="utf-8")
    json_calendar = Calendar.load(filename=str(filename))

json_calendar

## Optional calendar provider integrations

If you install the optional `bizdays[pmc]` extra, `Calendar.load(name="PMC/<calendar>")` can load calendars from `pandas_market_calendars`.

If you install the optional `bizdays[xcal]` extra, `Calendar.load(name="XCAL/<calendar>")` can load calendars from `exchange_calendars`.

If you install the optional `bizdays[work]` extra, `Calendar.load(name="WORK/<code>")` can load calendars from `workalendar`.

These are not packaged calendars; they come from optional integrations.

In [ ]:
{
    "pmc": Calendar.load(name="PMC/B3").name,
    "xcal": Calendar.load(name="XCAL/XNYS").name,
    "work": Calendar.load(name="WORK/FR").name,
}